In [1]:
import pandas as pd

<frozen importlib._bootstrap>:491: RuntimeWarning: The global interpreter lock (GIL) has been enabled to load module 'pandas._libs.pandas_parser', which has not declared that it can run safely without the GIL. To override this behavior and keep the GIL disabled (at your own risk), run with PYTHON_GIL=0 or -Xgil=0.


In [2]:
df = pd.read_csv("data/event_log_anonymized.csv").sort_values(["Case_id", "Completed On"])
df.drop_duplicates(["Case_id","Completed On"], keep = "first", inplace = True)

KeyboardInterrupt: 

In [ ]:
df.head()

,Recruiting Agency,Region,Country,Job Family,Job Family Group,All Stages for Candidate Current and Completed,Disposition Reason,Step,Event Record,Completed On,Completed By,CW,Evergreen,Rejected,hired,Job_Requisition_Code,Case_id
805943,NaN,4,15,57,16,Declined by Candidate\n\nInterview\n\nStart,Candidate Withdrew,Start,Review Candidate,2024-10-17T15:21:25.626,7566,0,0,0,0,10fb9be7d6ea,000086ee80c8 - 10fb9be7d6ea
805945,NaN,4,15,57,16,Declined by Candidate\n\nInterview\n\nStart,Candidate Withdrew,Provide Interview Rating and Comments,Provide Interview Rating and Comments,2024-10-17T15:22:10.434,6746,0,0,0,0,10fb9be7d6ea,000086ee80c8 - 10fb9be7d6ea
805949,NaN,4,15,57,16,Declined by Candidate\n\nInterview\n\nStart,Candidate Withdrew,Schedule Interview,Schedule Interview,2024-10-18T15:43:16.719,6746,0,0,0,0,10fb9be7d6ea,000086ee80c8 - 10fb9be7d6ea
805950,NaN,4,15,57,16,Declined by Candidate\n\nInterview\n\nStart,Candidate Withdrew,Manager Interview,Manager Interview,2024-10-24T15:44:32.706,7566,0,0,0,0,10fb9be7d6ea,000086ee80c8 - 10fb9be7d6ea
805953,NaN,4,15,57,16,Declined by Candidate\n\nInterview\n\nStart,Candidate Withdrew,Schedule Interview,Schedule Interview,2024-10-24T15:46:07.852,6746,0,0,0,0,10fb9be7d6ea,000086ee80c8 - 10fb9be7d6ea


In [ ]:
df['Candidate_id'] = df['Case_id'].str.extract(r'^(.*?)\s*-')

df = df[df["Completed On"]>="2021-01-01"]
grouping_cols = ['Job_Requisition_Code', 'Candidate_id', 'Recruiting Agency', 'Region', 'Country', 'Job Family',
       'Job Family Group','Case_id', 'CW', 'Evergreen', 'Rejected', 'hired','Completed By',"Step", "Disposition Reason","All Stages for Candidate Current and Completed"]
df = df.melt(id_vars = grouping_cols, value_vars = ["Completed On"], var_name = "Type", value_name = "Value")
df.drop("Type", axis = 1, inplace = True)
df.rename(columns = {"Value":"timestamp"}, inplace = True)
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [ ]:
import pm4py as pm
event_log = pm.convert_to_dataframe(df, case_id_key = "Case_id", timestamp_key = "timestamp", activity_key = "Step")

In [ ]:
event_log.to_csv("data/event_log_consolidated.csv", index = False)